# 83 Choosing among the candidates with execution (CPU runtime)

The generator's candidates (notebook 81) contain a correct answer far more often than the one we pick. This notebook tries better ways to pick, using only what is available when answering:

1. **Run every candidate once** the way it would run at answer time: compiled against the schema (no data read), then executed with a **1 second cap**, all candidates of a question in parallel. A query still running at 1 s gives no result.
2. **Pick by each method:**
   - `judge`: the candidate the judge rates most likely correct (as before);
   - `judge_compile`: the same, among the candidates that compile;
   - `result_vote`: candidates grouped by the result they return; the group written most often wins; its answer is the group's candidate the judge rates highest;
   - `weighted_vote`: the same, each written copy counting with the judge's p;
   - `result_vote_nonempty`, `weighted_vote_nonempty`: the same, but candidates that return no rows do not vote when some candidate returned rows.
   The method with the highest **validation** EX is kept as `vote_v1`. Dev and mini-dev are never used to choose.
3. **Score** every method on validation, dev and mini-dev with the usual evaluation, and **report** against the generator alone and the reference models.

**Runtime > Change runtime type > CPU.** About 1 hour, resumable: after a disconnect, run the setup and config cells, then the cell that was running.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
JUDGE = "judge_v1"        # the judge (notebook 80)
CAND = "judge_cand_v1"    # candidates, judge scores and labels (notebook 81)
VOTE = "vote_v1"          # answers per method (vote_v1_<method>) and the kept one (vote_v1)
GEN = "rl_v1_it8"         # the generator
METHODS = ["judge", "judge_compile", "result_vote", "weighted_vote", "result_vote_nonempty", "weighted_vote_nonempty"]
VAL_DBS = ",".join(json.load(open(f"{R}/sft_v1/export_meta.json"))["holdout_db_ids"])

### 1. Stage the databases and run every candidate with the 1 s cap

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev
!python -m effsql.judge exec --run {CAND} --splits val,dev,minidev

### 2. Pick by each method; keep the best on validation

Validation decides; dev and mini-dev are shown for the analysis only. `diff` and `p`: against the generator's own answer on the same questions. Oracle: some candidate is correct (the ceiling of any way of picking). The last line per split: how long a question waits for its candidates when they run in parallel.

In [ ]:
!python -m effsql.judge vote --run {CAND} --judge {JUDGE} --out {VOTE}

### 3. Score every method with the usual evaluation (resumable)

In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
for m in METHODS:
    r = f"{VOTE}_{m}"
    !python -m effsql.train eval-val --run {r}
    !effsql evaluate --pred {R}/{r}/predictions_dev.jsonl --split dev --timing none --run {r}_dev --model {MODEL} --method {m}
    !effsql evaluate --pred {R}/{r}/predictions_minidev.jsonl --split minidev --timing none --run {r}_minidev --model {MODEL} --method {m}

### 4. Report: accuracy against the generator alone and the reference models

Saved as `vote_v1_report/report.txt`.

In [ ]:
RUNS = ["pool_v1_{split}_base", GEN] + [f"{VOTE}_{m}" for m in METHODS] + ["ref_omnisql7b", "ref_arctic7b", "ref_qwen32b"]
LABELS = ",".join(["pool_v1_{split}_base=zero-shot 7B", f"{GEN}=generator alone (RL)", f"{VOTE}_judge=judge",
                   f"{VOTE}_judge_compile=judge + compile check", f"{VOTE}_result_vote=result vote",
                   f"{VOTE}_weighted_vote=judge-weighted result vote",
                   f"{VOTE}_result_vote_nonempty=result vote (rows first)",
                   f"{VOTE}_weighted_vote_nonempty=judge-weighted vote (rows first)", "ref_omnisql7b=OmniSQL-7B",
                   "ref_arctic7b=Arctic-R1-7B", "ref_qwen32b=Qwen-32B zero-shot"])
assert all("," not in x.split("=", 1)[1] for x in LABELS.split(",")) and LABELS.count("=") == len(RUNS)
!python -m effsql.train report --runs "{','.join(RUNS)}" --ref {GEN} --labels "{LABELS}" --out vote_v1_report

### Done: disconnect the runtime

Then run `84_latency` (A100) to time the whole answer path question by question.